In [1]:
import os
import json
import time
import pandas as pd
from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), '.env'))
API_KEY = os.getenv("YOUTUBE_API_KEY")
youtube = build("youtube", "v3", developerKey=API_KEY)

print("API ready:", API_KEY is not None)

API ready: True


In [2]:
VIDEOS = {
    "ai_slop_renaissance":        "0iT9HbaRwfM",
    "replacing_humans_wrong":     "QX1Xwzm9yHY",
    "replacing_developers_wrong": "WfjGZCuxl-U",
    "ai_might_not_replace_jobs":  "EGskcTRnLJ0",
    "ai_white_collar_12months":   "0lwftepuvYA",
    "ai_fails_96_percent":        "z3kaLM8Oj4o",
    "who_buys_everything":        "MYB0SVTGRj4",
    "capitalism_no_workers":      "yhpyHV1iz00",
}

print(f"{len(VIDEOS)} videos loaded")

8 videos loaded


In [9]:
def scrape_comments(video_id, video_name, max_comments=8000):
    comments = []
    next_page_token = None

    while len(comments) < max_comments:
        try:
            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=100,
                pageToken=next_page_token,
                textFormat="plainText"
            )
            response = request.execute()
        except Exception as e:
            print(f" Error on {video_name}: {e}")
            break

        for item in response["items"]:
            snippet = item["snippet"]["topLevelComment"]["snippet"]
            comments.append({
                "video_id": video_id,
                "video_name": video_name,
                "comment_id": item["snippet"]["topLevelComment"]["id"],
                "author": snippet["authorDisplayName"],
                "text": snippet["textDisplay"],
                "likes": snippet["likeCount"],
                "published_at": snippet["publishedAt"],
                "reply_count": item["snippet"]["totalReplyCount"]
            })

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

        time.sleep(0.3)

    return comments

print("scrape_comments function defined")

scrape_comments function defined


In [11]:
def scrape_transcript(video_id, video_name):
    try:
        ytt = YouTubeTranscriptApi()
        transcript_list = ytt.fetch(video_id)
        full_text = " ".join([chunk.text for chunk in transcript_list])
        return {
            "video_id":   video_id,
            "video_name": video_name,
            "transcript": full_text,
            "chunks":     len(transcript_list)
        }
    except Exception as e:
        print(f"  No transcript for {video_name}: {e}")
        return None

print("scrape_transcript function defined")

scrape_transcript function defined


In [12]:
all_comments = []
all_transcripts = []

for video_name, video_id in VIDEOS.items():
    print(f"\nScraping: {video_name}")
    
    # comments
    comments = scrape_comments(video_id, video_name, max_comments=8000)
    all_comments.extend(comments)
    print(f"  comments collected: {len(comments)}")
    
    # transcript
    transcript = scrape_transcript(video_id, video_name)
    if transcript:
        all_transcripts.append(transcript)
        print(f"  transcript chunks: {transcript['chunks']}")
    
    time.sleep(1)  # pause between videos

print(f"\nDone.")
print(f"Total comments : {len(all_comments)}")
print(f"Total transcripts: {len(all_transcripts)}")


Scraping: ai_slop_renaissance
  comments collected: 5096
  transcript chunks: 121

Scraping: replacing_humans_wrong
  comments collected: 3747
  transcript chunks: 408

Scraping: replacing_developers_wrong
  comments collected: 5242
  transcript chunks: 207

Scraping: ai_might_not_replace_jobs
  comments collected: 681
  transcript chunks: 304

Scraping: ai_white_collar_12months
  comments collected: 923
  transcript chunks: 316

Scraping: ai_fails_96_percent
  comments collected: 3620
  transcript chunks: 361

Scraping: who_buys_everything
  comments collected: 7205
  transcript chunks: 148

Scraping: capitalism_no_workers
  comments collected: 1946
  transcript chunks: 194

Done.
Total comments : 28460
Total transcripts: 8


In [13]:
os.makedirs("../data/raw", exist_ok=True)

with open("../data/raw/comments_raw.json", "w", encoding="utf-8") as f:
    json.dump(all_comments, f, indent=2, ensure_ascii=False)

with open("../data/raw/transcripts_raw.json", "w", encoding="utf-8") as f:
    json.dump(all_transcripts, f, indent=2, ensure_ascii=False)

print(f"Saved {len(all_comments)} comments to data/raw/comments_raw.json")
print(f"Saved {len(all_transcripts)} transcripts to data/raw/transcripts_raw.json")

Saved 28460 comments to data/raw/comments_raw.json
Saved 8 transcripts to data/raw/transcripts_raw.json


In [15]:
df_comments = pd.DataFrame(all_comments)
df_transcripts = pd.DataFrame(all_transcripts)

print("Comments shape:", df_comments.shape)
print("Transcripts shape:", df_transcripts.shape)
print("\nComment columns:", df_comments.columns.tolist())
print("\nSample comment:")
print(df_comments["text"].iloc[0])
print("\nComments per video:")
print(df_comments["video_name"].value_counts())

Comments shape: (28460, 8)
Transcripts shape: (8, 4)

Comment columns: ['video_id', 'video_name', 'comment_id', 'author', 'text', 'likes', 'published_at', 'reply_count']

Sample comment:
Wow. It blows my mind the amount of commenters who thought that these videos were made by Ai or a software program. They've always been drawn by hand on an actual whiteboard with dry-erase markers. I would probably use Ai, but the smell of the markers is just too good. JK! There's almost nothing I love more than drawing - It's peaceful, fun and helps you learn a lot about yourself. The irony of Ai is that it's pushing many humans back into real world hobbies again - playing instruments, climbing trees, wood carving, welding, sports, reading physical books, etc. All this slop has made me appreciate the real world so much more.

Comments per video:
video_name
who_buys_everything           7205
replacing_developers_wrong    5242
ai_slop_renaissance           5096
replacing_humans_wrong        3747
ai_fail

In [16]:
print("=== Before Cleaning ===")
print("Total comments:", len(df_comments))
print("Duplicate texts:", df_comments.duplicated(subset="text").sum())
print("Comments under 15 chars:", (df_comments["text"].str.len() < 15).sum())

# drop duplicates
df_comments = df_comments.drop_duplicates(subset="text")

# drop short comments
df_comments = df_comments[df_comments["text"].str.len() >= 15]

# reset index
df_comments = df_comments.reset_index(drop=True)

print("\n=== After Cleaning ===")
print("Total comments:", len(df_comments))
print("Comments per video:")
print(df_comments["video_name"].value_counts())

=== Before Cleaning ===
Total comments: 28460
Duplicate texts: 280
Comments under 15 chars: 997

=== After Cleaning ===
Total comments: 27330
Comments per video:
video_name
who_buys_everything           6996
replacing_developers_wrong    4992
ai_slop_renaissance           4719
replacing_humans_wrong        3639
ai_fails_96_percent           3528
capitalism_no_workers         1894
ai_white_collar_12months       900
ai_might_not_replace_jobs      662
Name: count, dtype: int64


In [17]:
os.makedirs("../data/processed", exist_ok=True)

df_comments.to_csv("../data/processed/comments_clean.csv", index=False, encoding="utf-8-sig")
df_transcripts.to_csv("../data/processed/transcripts_raw.csv", index=False, encoding="utf-8-sig")

print(f"Saved {len(df_comments)} clean comments to data/processed/comments_clean.csv")
print(f"Saved {len(df_transcripts)} transcripts to data/processed/transcripts_raw.csv")

Saved 27330 clean comments to data/processed/comments_clean.csv
Saved 8 transcripts to data/processed/transcripts_raw.csv
